# Judging Books by Their Covers

In [ ]:
import pandas as pd

df = pd.read_csv('data/books.csv', sep=';', encoding='latin1', escapechar='\\')
books_features = pd.read_csv('data/books_features.csv', low_memory=False)
ratings = pd.read_csv('data/ratings.csv', sep=';', encoding='latin1', escapechar='\\')
users = pd.read_csv('data/users.csv', sep=';', encoding='latin1', escapechar='\\')

sample = df.sample(300)
print(len(df))

# sample.to_csv('data/book_sample.csv', index=False, sep=';', encoding='latin1', escapechar='\\')

df.head()

Extracted features. 96285 errors? All 1x1 pixel images?

In [ ]:
books_features.head()



## Merge and clean datasets

In [ ]:
# Rename 'isbn' to 'ISBN' in books_features to match the main dataframe
books_features = books_features.rename(columns={'isbn': 'ISBN'})

# Merge the books data with the extracted cover features
merged_df = pd.merge(df, books_features, on='ISBN', how='inner')

# Drop the 1x1 pixel image errors you identified
merged_df = merged_df[merged_df['error'].isna()]

# Ensure we only keep rows that successfully extracted all three colors
merged_df = merged_df.dropna(subset=['color1_hex', 'color2_hex', 'color3_hex'])

print(f"Total usable books: {len(merged_df)}")
merged_df[['ISBN', 'Book-Title', 'color1_hex', 'color2_hex', 'color3_hex']].head()

## Convert colors to usable numbers for bayes

In [ ]:
import colorsys

def hex_to_hsv(hex_color):
    """Converts a hex color string (e.g., '#baa68e') to H, S, V components."""
    if pd.isna(hex_color):
        return pd.Series([None, None, None])
    
    # Remove '#' and convert to RGB values between 0 and 1
    hex_color = str(hex_color).lstrip('#')
    r, g, b = tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))
    
    # Convert to HSV
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    return pd.Series([h, s, v])

# Apply the conversion function to each of the three color columns
print("Converting Color 1...")
merged_df[['H1', 'S1', 'V1']] = merged_df['color1_hex'].apply(hex_to_hsv)

print("Converting Color 2...")
merged_df[['H2', 'S2', 'V2']] = merged_df['color2_hex'].apply(hex_to_hsv)

print("Converting Color 3...")
merged_df[['H3', 'S3', 'V3']] = merged_df['color3_hex'].apply(hex_to_hsv)

print("Conversion complete!")
merged_df[['ISBN', 'Book-Title', 'H1', 'S1', 'V1', 'H2', 'S2', 'V2']].head()

In [ ]:
# 1. Convert ISBN-10 to ISBN-13 so the datasets can actually match
def isbn10_to_isbn13(isbn10):
    isbn = str(isbn10).strip().upper()
    if len(isbn) != 10:
        return isbn 
        
    core = '978' + isbn[:-1]
    
    total = 0
    for i, digit in enumerate(core):
        if digit.isdigit():
            weight = 1 if i % 2 == 0 else 3
            total += int(digit) * weight
        else:
            return isbn 
            
    check_digit = 10 - (total % 10)
    if check_digit == 10:
        check_digit = 0
        
    return core + str(check_digit)

print("Converting ISBN-10s to ISBN-13s...")
merged_df['ISBN13'] = merged_df['ISBN'].apply(isbn10_to_isbn13)

# 2. Merge using the converted ISBNs
overlap_df = pd.merge(merged_df, best_books_df, left_on='ISBN13', right_on='ISBN', how='inner')
print(f"Total overlapping books between datasets: {len(overlap_df)}")

# 3. Clean and map the genres
genre_col = 'genres' if 'genres' in overlap_df.columns else 'genre'
overlap_df['Genre'] = overlap_df[genre_col].apply(clean_genre)

# 4. Drop missing and 'Other' categories, keeping all remaining books
df_model = overlap_df.dropna(subset=['Genre'])
df_model = df_model[df_model['Genre'] != 'Other'].reset_index(drop=True)

print(f"\nFinal usable books for the model: {len(df_model)}")
print("\nFull Genre Distribution:")
print(df_model['Genre'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Define our features (X) and target label (y)
feature_cols = ['H1', 'S1', 'V1', 'H2', 'S2', 'V2', 'H3', 'S3', 'V3']
X = df_model[feature_cols]
y = df_model['Genre']

# 2. Split the dataset into training and testing sets (80% train, 20% test)
# random_state ensures we get the same split every time we run it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize the Gaussian Naive Bayes classifier
gnb = GaussianNB()

# 4. Train the model using our training data
gnb.fit(X_train, y_train)

# 5. Make predictions on the test set
y_pred = gnb.predict(X_test)

# 6. Evaluate and print the results
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
# zero_division=0 prevents warnings if a small class isn't predicted
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=gnb.classes_)

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=gnb.classes_, yticklabels=gnb.classes_)
plt.xlabel('Guessed Genre (Predicted)')
plt.ylabel('Actual Genre (True)')
plt.title('Gaussian Naive Bayes: Actual vs Guessed Book Genres based on Cover Colors')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualizing the distribution of the Primary Hue (H1) to show why Gaussian was chosen
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_model, x='H1', hue='Genre', common_norm=False, fill=True, alpha=0.3)
plt.title('Distribution of Primary Cover Hue (H1) by Genre')
plt.xlabel('Hue (H1) - Continuous Value (0.0 to 1.0)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()